# 5단계: 평가
평가 코드는 `src/evaluate.py` 에 있다. 이 노트북은 GitHub 에서 코드를 받아 **실행만** 한다.
(코드를 고칠 땐 랩탑에서 고치고 push → 여기서 `git pull`)

- 런타임: **A100 GPU** (생성만 하므로 40GB 도 충분). 🔑 Secrets에 `HF_TOKEN` 등록
- 생성 결과(`preds.jsonl`)와 채점 결과는 Drive 의 `MyDrive/octopus_v2/runs/eval/` 에 저장 → 랩탑에서도 다시 채점 가능
- 생성이 중간에 끊기면 같은 칸을 다시 실행하면 남은 것만 이어서 한다
- 비교 기준: 논문 표 1 Octopus-3 (함수당 100개 + full) 정확도 **98.095%**, 지연 **0.38초** (A100, 양자화 없음)

## 준비 (그냥 실행)

In [1]:
# 랩탑에서 확인한 버전으로 고정 (최신 5.x 는 API 가 바뀌어 에러가 난다)
# 처음 설치한 뒤에는 '런타임 > 세션 다시 시작' 후 이 칸부터 다시 실행
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive
Cloning into '/content/octopus_v2_reproduce'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 152 (delta 78), reused 98 (delta 32), pack-reused 0 (

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers; print("transformers", transformers.__version__)  # 4.56.2 여야 함

RUN_DIR = "/content/drive/MyDrive/octopus_v2/runs/full"   # 4단계에서 학습한 모델
EVAL = "/content/drive/MyDrive/octopus_v2/runs/eval"

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
transformers 4.56.2


---
## 5-1. 우리 모델로 test 800개 생성
**예측**: 전체 정확도는 논문 98.095% 보다 높을까 낮을까? 틀린다면 어느 함수에서?
> (여기에 예측)

처음 3개와 100개마다 생성 결과가 찍힌다. `<nexa_end>` 에서 멈추는지 본다.

In [3]:
!python -m src.evaluate generate --model {RUN_DIR} --out {EVAL}/ours/preds.jsonl

[생성] 800개 중 이미 0개 있음 → 800개 생성 → /content/drive/MyDrive/octopus_v2/runs/eval/ours/preds.jsonl
2026-10-04 14:07:25.606502: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-04 14:07:25.685259: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading checkpoint shards: 100% 2/2 [01:07<00:00, 33.67s/it]
[설정] {'model': '/content/drive/MyDrive/octopus_v2/runs/full', 'tiny': False, 'test_file': 'data/test.jsonl', 'limit': None, 'max_new_tokens': 128, 'warmup': 5, 'dtype': 'torch.bfloat16', 'attn_implementation': 'sdpa', '

In [4]:
!python -m src.evaluate score --preds {EVAL}/ours/preds.jsonl

                                                      전체   seen_in_train=false
개수                                                   800                   768
형식 OK                                            99.50%                99.48% 
함수 선택 정확도                                        99.38%                99.35% 
인자 정확도 (함수 맞힌 positive 중)                        89.65%                88.74% 
전체 정확도 ★                                         94.25%                94.01% 
  positive                                       88.75%                87.77% 
  negative                                       99.75%                99.75% 
느슨한 정확도 (논문에 없는 추가)                              96.12%                95.96% 
호출 문자열 완전 일치                                     94.25%                94.01% 

[결과 종류별 개수]
  args:count_mismatch                                    2                     2
  args:valid_but_different                              39                    39
  correct                          

---
## 5-2. (논문에 없는 추가) 공개 모델 `NexaAIDev/Octopus-v2` 로 같은 test 생성
기준점. 우리 모델이 틀린 문제를 공개 모델도 틀리면 데이터가 어려운 것, 공개 모델만 맞히면 학습 쪽 문제.
공개 모델이 어떤 데이터로 학습했는지는 모르므로 참고용.

In [5]:
!python -m src.evaluate generate --model NexaAIDev/Octopus-v2 --out {EVAL}/public/preds.jsonl

[생성] 800개 중 이미 0개 있음 → 800개 생성 → /content/drive/MyDrive/octopus_v2/runs/eval/public/preds.jsonl
tokenizer_config.json: 6.35kB [00:00, 13.4MB/s]
tokenizer.model: 100% 4.24M/4.24M [00:01<00:00, 4.07MB/s]
tokenizer.json: 100% 17.5M/17.5M [00:00<00:00, 28.6MB/s]
added_tokens.json: 100% 500/500 [00:00<00:00, 4.85MB/s]
special_tokens_map.json: 100% 957/957 [00:00<00:00, 8.69MB/s]
config.json: 100% 673/673 [00:00<00:00, 6.77MB/s]
2026-10-04 14:12:24.282168: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-04 14:12:24.360717: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, reb

In [6]:
!python -m src.evaluate score --preds {EVAL}/public/preds.jsonl

                                                      전체   seen_in_train=false
개수                                                   800                   768
형식 OK                                            96.38%                96.35% 
함수 선택 정확도                                        66.50%                65.36% 
인자 정확도 (함수 맞힌 positive 중)                        60.53%                59.13% 
전체 정확도 ★                                         48.00%                47.01% 
  positive                                       56.75%                55.43% 
  negative                                       39.25%                39.25% 
느슨한 정확도 (논문에 없는 추가)                              52.00%                50.65% 
호출 문자열 완전 일치                                     48.00%                47.01% 

[결과 종류별 개수]
  args:count_mismatch                                   16                    16
  args:rule_violation                                   29                    28
  args:valid_but_different         

---
## 5-2b. (논문에 없는 추가) 우리 모델·공개 모델이 고른 토큰의 softmax 확률
`preds.jsonl` 에는 생성된 글만 있다. `src/probs.py` 로 같은 조건(greedy)에서 **다시 생성**하면서
test 전체(맞은 것도 포함)의 단계마다 상위 5개 토큰과 확률을 `{EVAL}/ours/probs.jsonl`, `{EVAL}/public/probs.jsonl` 에 저장한다. 지연 시간은 안 잰다.

- 출력이 공백으로 시작하면 첫 단계는 공백 토큰(`▁`)이다. 그래서 **처음 함수 토큰을 고른 단계**를 따로 `func` 에 적는다.
- `func.p_chosen`: 고른 함수 토큰의 확률 / `func.p_irrelevant`: 같은 자리에서 `<nexa_20>`(irrelevant_function) 의 확률
- `same_as_preds` 가 모두 True 면 5-1·5-2 와 같은 출력 (그 채점 결과를 그대로 붙여 볼 수 있다)
- 두 모델의 false_call 을 비교: 같은 질문에서 둘 다 확신하며 틀리면 질문·라벨 쪽, 한쪽만 그러면 학습 쪽을 의심 (추정)

In [ ]:
# 우리 모델
!python -m src.probs generate --model {RUN_DIR} --preds {EVAL}/ours/preds.jsonl --out {EVAL}/ours/probs.jsonl

In [ ]:
!python -m src.probs report --probs {EVAL}/ours/probs.jsonl --scored {EVAL}/ours/scored.jsonl

In [ ]:
# 공개 모델
!python -m src.probs generate --model NexaAIDev/Octopus-v2 --preds {EVAL}/public/preds.jsonl --out {EVAL}/public/probs.jsonl

In [ ]:
!python -m src.probs report --probs {EVAL}/public/probs.jsonl --scored {EVAL}/public/scored.jsonl

---
## 5-3. 나란히 보기
오답 목록은 `{EVAL}/ours/errors.csv` 를 랩탑에서 열어 `review` 칸에 메모한다.

In [7]:
import json
from src.evaluate import METRIC_NAMES

rows = {name: json.load(open(f"{EVAL}/{name}/summary.json")) for name in ("ours", "public")}
print(f"{'':<36}{'ours':>12}{'public':>12}")
for sub in ("전체", "seen_in_train=false"):
    print(f"[{sub}]")
    for key, label in METRIC_NAMES.items():
        cells = [rows[n][sub][key] for n in rows]
        print(f"  {label:<34}" + "".join(f"{c:>12}" if key == "n" else f"{c:>12.2%}" for c in cells))
print("[지연 시간, 초]")
for k in ("median", "p90"):
    print(f"  {k:<34}" + "".join(f"{rows[n]['latency_s'][k]:>12.3f}" for n in rows))

                                            ours      public
[전체]
  개수                                         800         800
  형식 OK                                   99.50%      96.38%
  함수 선택 정확도                               99.38%      66.50%
  인자 정확도 (함수 맞힌 positive 중)               89.65%      60.53%
  전체 정확도 ★                                94.25%      48.00%
    positive                              88.75%      56.75%
    negative                              99.75%      39.25%
  느슨한 정확도 (논문에 없는 추가)                     96.12%      52.00%
  호출 문자열 완전 일치                            94.25%      48.00%
[seen_in_train=false]
  개수                                         768         768
  형식 OK                                   99.48%      96.35%
  함수 선택 정확도                               99.35%      65.36%
  인자 정확도 (함수 맞힌 positive 중)               88.74%      59.13%
  전체 정확도 ★                                94.01%      47.01%
    positive                              87.77%      55.4